In [ ]:
import pandas as pd
import pycountry
import json
from os import listdir as ls
from datetime import datetime, timedelta
from matplotlib import pyplot as plt 

from emu_renewal.inputs import DATA_PATH, get_apple_mobility, get_worldbank_national_pop, \
    get_country_vacc_data, get_indicator_series_from_who_data, get_country_vacc_data
from emu_renewal.run import find_run_start_time, find_run_end_time

In [ ]:
pop_threshold = 1e6
gmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "gmob" in i]
fbmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "fbmob" in i]
amob_avail = []
amob_unavail = []
for c in pycountry.countries:
    try:
        iso3 = c.alpha_3
        get_apple_mobility(iso3)
        amob_avail.append(iso3)
    except:
        amob_unavail.append(iso3)
any_mob_avail = list(set(amob_avail + fbmob_avail + gmob_avail))

In [ ]:
cases_data = {}
deaths_data = {}
pops = {}
for c in any_mob_avail:
    try:
        pops[c] = get_worldbank_national_pop(iso3)
    except:
        pops[c] = get_undesa_national_pop(iso3)
    cases_data[c] = get_indicator_series_from_who_data("New_cases", c)
    deaths_data[c] = get_indicator_series_from_who_data("New_deaths", c)

In [ ]:
def plot_cases_deaths(country, start, end, per_capita=True):
    c_cases = cases_data[country][(start < country_cases.index) & (country_cases.index < end)]
    c_deaths = deaths_data[country][(start < country_deaths.index) & (country_deaths.index < end)]
    fig, axes = plt.subplots(2, 1, sharex=True)
    title_end = ", per capita" if per_capita else ""
    data = c_cases / pops[country] if per_capita else c_cases
    axes[0].plot(data.index, data, marker="o", linewidth=0.0, markersize=3.0)
    axes[0].set_title(f"cases{title_end}")
    data = c_deaths / pops[country] if per_capita else c_deaths
    axes[1].plot(data.index, data, marker="o", linewidth=0.0, markersize=3.0)
    axes[1].set_title(f"deaths{title_end}")
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=70)
    fig.tight_layout()

In [ ]:
plot_cases_deaths("LUX", datetime(2020, 1, 1), datetime(2021, 6, 30), False)